# SASRec BPI2012 Colab Train (Eval Fix)

Colab notebook for re-running the selected SASRec baseline settings after fixing the evaluation code.

This notebook runs both model-selection criteria:
- `full_valid_ndcg@10`
- `full_valid_ndcg@5`

Output directories:
- `sasrec_bpi2012_ndcg10`
- `sasrec_bpi2012_ndcg5`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('NDCG10_OUTPUT_DIR:', NDCG10_OUTPUT_DIR)
print('NDCG5_OUTPUT_DIR:', NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$NDCG10_OUTPUT_DIR"
!mkdir -p "$NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction


/content
/content/time-aware-behavior-prediction


In [6]:
# If you need the latest code from GitHub, uncomment below.
%cd /content/time-aware-behavior-prediction
!git pull


/content/time-aware-behavior-prediction
Already up to date.


In [7]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [8]:
!pip install -r requirements_colab.txt


In [9]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [10]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Planned runs

Selected baseline runs to re-run after the evaluation fix:
- `anchor_pd_s42`
- `anchor_ml100_s42`
- `anchor_ml50_s42`
- `anchor_ml20_s42`
- `refine_ml50_do030_s42`
- `refine_ml50_do035_s42`
- `refine_ml50_do025_s42`
- `refine_ml50_do025_s2024`
- `refine_ml50_do030_s2024`
- `refine_ml75_do030_s42`
- `refine_ml100_do030_s42`


In [11]:
from pathlib import Path

planned_run_names = [
    'anchor_pd_s42',
    'anchor_ml100_s42',
    'anchor_ml50_s42',
    'anchor_ml20_s42',
    'refine_ml50_do030_s42',
    'refine_ml50_do035_s42',
    'refine_ml50_do025_s42',
    'refine_ml50_do025_s2024',
    'refine_ml50_do030_s2024',
    'refine_ml75_do030_s42',
    'refine_ml100_do030_s42',
]

for label, output_dir in [('NDCG@10', Path(NDCG10_OUTPUT_DIR)), ('NDCG@5', Path(NDCG5_OUTPUT_DIR))]:
    print('=' * 80)
    print(label)
    for run_name in planned_run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


NDCG@10
anchor_pd_s42 OK
anchor_ml100_s42 OK
anchor_ml50_s42 OK
anchor_ml20_s42 OK
refine_ml50_do030_s42 OK
refine_ml50_do035_s42 OK
refine_ml50_do025_s42 OK
refine_ml50_do025_s2024 OK
refine_ml50_do030_s2024 OK
refine_ml75_do030_s42 OK
refine_ml100_do030_s42 OK
NDCG@5
anchor_pd_s42 OK
anchor_ml100_s42 OK
anchor_ml50_s42 OK
anchor_ml20_s42 OK
refine_ml50_do030_s42 OK
refine_ml50_do035_s42 OK
refine_ml50_do025_s42 OK
refine_ml50_do025_s2024 OK
refine_ml50_do030_s2024 OK
refine_ml75_do030_s42 OK
refine_ml100_do030_s42 OK


## Train runs with `selection_metric = full_valid_ndcg@10`


### anchor_pd_s42


In [12]:
!python src/train_sasrec.py \
  --run_name anchor_pd_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 200 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_pd_s42
epoch=1, loss=0.4974
epoch=2, loss=0.2282
epoch=3, loss=0.1655
epoch=4, loss=0.1360
epoch=5, loss=0.1199
valid [full], NDCG@5: 0.6457, HR@5: 0.7414, NDCG@10: 0.7226, HR@10: 0.9828, MRR: 0.6469
valid [sampled], NDCG@5: 0.5632, HR@5: 0.5772, NDCG@10: 0.5810, HR@10: 0.6324, MRR: 0.5772
test [full], NDCG@5: 0.5359, HR@5: 0.7313, NDCG@10: 0.6238, HR@10: 1.0000, MRR: 0.5074
test [sampled], NDCG@5: 0.1698, HR@5: 0.1750, NDCG@10: 0.2127, HR@10: 0.3140, MRR: 0.2129
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/

### anchor_ml100_s42


In [13]:
!python src/train_sasrec.py \
  --run_name anchor_ml100_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 100 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_ml100_s42
epoch=1, loss=0.4891
epoch=2, loss=0.2275
epoch=3, loss=0.1620
epoch=4, loss=0.1320
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.6230, HR@5: 0.6725, NDCG@10: 0.7174, HR@10: 0.9646, MRR: 0.6482
valid [sampled], NDCG@5: 0.5689, HR@5: 0.5814, NDCG@10: 0.5844, HR@10: 0.6298, MRR: 0.5812
test [full], NDCG@5: 0.4660, HR@5: 0.6063, NDCG@10: 0.5864, HR@10: 0.9842, MRR: 0.4681
test [sampled], NDCG@5: 0.1427, HR@5: 0.1480, NDCG@10: 0.1838, HR@10: 0.2807, MRR: 0.1835
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outpu

### anchor_ml50_s42


In [14]:
!python src/train_sasrec.py \
  --run_name anchor_ml50_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_ml50_s42
epoch=1, loss=0.4793
epoch=2, loss=0.2166
epoch=3, loss=0.1564
epoch=4, loss=0.1245
epoch=5, loss=0.1127
valid [full], NDCG@5: 0.6566, HR@5: 0.7705, NDCG@10: 0.7221, HR@10: 0.9808, MRR: 0.6467
valid [sampled], NDCG@5: 0.5561, HR@5: 0.5710, NDCG@10: 0.5758, HR@10: 0.6326, MRR: 0.5713
test [full], NDCG@5: 0.4875, HR@5: 0.6521, NDCG@10: 0.5959, HR@10: 0.9999, MRR: 0.4741
test [sampled], NDCG@5: 0.1370, HR@5: 0.1437, NDCG@10: 0.1829, HR@10: 0.2915, MRR: 0.1796
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/output

### anchor_ml20_s42


In [15]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/anchor_ml20_s42
epoch=1, loss=0.6703
epoch=2, loss=0.3156
epoch=3, loss=0.2178
epoch=4, loss=0.1735
epoch=5, loss=0.1464
valid [full], NDCG@5: 0.7050, HR@5: 0.8805, NDCG@10: 0.7424, HR@10: 0.9916, MRR: 0.6648
valid [sampled], NDCG@5: 0.5452, HR@5: 0.5520, NDCG@10: 0.5650, HR@10: 0.6151, MRR: 0.5682
test [full], NDCG@5: 0.8392, HR@5: 0.9371, NDCG@10: 0.8599, HR@10: 1.0000, MRR: 0.8146
test [sampled], NDCG@5: 0.1971, HR@5: 0.2797, NDCG@10: 0.2821, HR@10: 0.5416, MRR: 0.2314
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/output

### refine_ml50_do030_s42


In [16]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do030_s42
epoch=1, loss=0.5360
epoch=2, loss=0.2475
epoch=3, loss=0.1849
epoch=4, loss=0.1493
epoch=5, loss=0.1305
valid [full], NDCG@5: 0.6477, HR@5: 0.7576, NDCG@10: 0.6943, HR@10: 0.9076, MRR: 0.6380
valid [sampled], NDCG@5: 0.5544, HR@5: 0.5666, NDCG@10: 0.5702, HR@10: 0.6159, MRR: 0.5690
test [full], NDCG@5: 0.4378, HR@5: 0.5577, NDCG@10: 0.5739, HR@10: 0.9914, MRR: 0.4509
test [sampled], NDCG@5: 0.1360, HR@5: 0.1425, NDCG@10: 0.1793, HR@10: 0.2819, MRR: 0.1759
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml50_do035_s42


In [17]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do035_s42
epoch=1, loss=0.5673
epoch=2, loss=0.2638
epoch=3, loss=0.1987
epoch=4, loss=0.1638
epoch=5, loss=0.1414
valid [full], NDCG@5: 0.6434, HR@5: 0.7559, NDCG@10: 0.6854, HR@10: 0.8896, MRR: 0.6333
valid [sampled], NDCG@5: 0.5545, HR@5: 0.5603, NDCG@10: 0.5654, HR@10: 0.5947, MRR: 0.5698
test [full], NDCG@5: 0.4390, HR@5: 0.6201, NDCG@10: 0.5094, HR@10: 0.8432, MRR: 0.4211
test [sampled], NDCG@5: 0.1322, HR@5: 0.1359, NDCG@10: 0.1472, HR@10: 0.1844, MRR: 0.1664
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml50_do025_s42


In [18]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do025_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do025_s42
epoch=1, loss=0.5066
epoch=2, loss=0.2315
epoch=3, loss=0.1700
epoch=4, loss=0.1361
epoch=5, loss=0.1203
valid [full], NDCG@5: 0.6551, HR@5: 0.7708, NDCG@10: 0.7204, HR@10: 0.9803, MRR: 0.6446
valid [sampled], NDCG@5: 0.5541, HR@5: 0.5661, NDCG@10: 0.5730, HR@10: 0.6257, MRR: 0.5705
test [full], NDCG@5: 0.4666, HR@5: 0.6195, NDCG@10: 0.5856, HR@10: 1.0000, MRR: 0.4610
test [sampled], NDCG@5: 0.1341, HR@5: 0.1405, NDCG@10: 0.1812, HR@10: 0.2923, MRR: 0.1766
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml50_do025_s2024


In [19]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do025_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do025_s2024
epoch=1, loss=0.5681
epoch=2, loss=0.2402
epoch=3, loss=0.1763
epoch=4, loss=0.1409
epoch=5, loss=0.1248
valid [full], NDCG@5: 0.6843, HR@5: 0.8465, NDCG@10: 0.7338, HR@10: 0.9939, MRR: 0.6549
valid [sampled], NDCG@5: 0.5604, HR@5: 0.5637, NDCG@10: 0.5685, HR@10: 0.5894, MRR: 0.5784
test [full], NDCG@5: 0.7561, HR@5: 0.8822, NDCG@10: 0.7931, HR@10: 1.0000, MRR: 0.7288
test [sampled], NDCG@5: 0.1681, HR@5: 0.2351, NDCG@10: 0.2364, HR@10: 0.4463, MRR: 0.2037
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-predictio

### refine_ml50_do030_s2024


In [20]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do030_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml50_do030_s2024
epoch=1, loss=0.5987
epoch=2, loss=0.2553
epoch=3, loss=0.1894
epoch=4, loss=0.1533
epoch=5, loss=0.1350
valid [full], NDCG@5: 0.7439, HR@5: 0.9658, NDCG@10: 0.7536, HR@10: 0.9931, MRR: 0.6785
valid [sampled], NDCG@5: 0.5670, HR@5: 0.5725, NDCG@10: 0.5807, HR@10: 0.6159, MRR: 0.5882
test [full], NDCG@5: 0.7688, HR@5: 0.9350, NDCG@10: 0.7900, HR@10: 1.0000, MRR: 0.7235
test [sampled], NDCG@5: 0.1627, HR@5: 0.2252, NDCG@10: 0.2248, HR@10: 0.4173, MRR: 0.1993
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-predictio

### refine_ml75_do030_s42


In [21]:
!python src/train_sasrec.py \
  --run_name refine_ml75_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 75 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml75_do030_s42
epoch=1, loss=0.5423
epoch=2, loss=0.2566
epoch=3, loss=0.1895
epoch=4, loss=0.1550
epoch=5, loss=0.1346
valid [full], NDCG@5: 0.6334, HR@5: 0.7080, NDCG@10: 0.7187, HR@10: 0.9788, MRR: 0.6444
valid [sampled], NDCG@5: 0.5632, HR@5: 0.5782, NDCG@10: 0.5810, HR@10: 0.6336, MRR: 0.5759
test [full], NDCG@5: 0.4215, HR@5: 0.5434, NDCG@10: 0.5647, HR@10: 0.9999, MRR: 0.4357
test [sampled], NDCG@5: 0.1462, HR@5: 0.1517, NDCG@10: 0.1850, HR@10: 0.2766, MRR: 0.1841
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml100_do030_s42


In [22]:
!python src/train_sasrec.py \
  --run_name refine_ml100_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 100 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10/refine_ml100_do030_s42
epoch=1, loss=0.5443
epoch=2, loss=0.2581
epoch=3, loss=0.1902
epoch=4, loss=0.1567
epoch=5, loss=0.1356
valid [full], NDCG@5: 0.6461, HR@5: 0.7348, NDCG@10: 0.7279, HR@10: 0.9925, MRR: 0.6507
valid [sampled], NDCG@5: 0.5705, HR@5: 0.5853, NDCG@10: 0.5859, HR@10: 0.6330, MRR: 0.5827
test [full], NDCG@5: 0.4594, HR@5: 0.5923, NDCG@10: 0.5847, HR@10: 0.9859, MRR: 0.4657
test [sampled], NDCG@5: 0.1540, HR@5: 0.1590, NDCG@10: 0.1933, HR@10: 0.2859, MRR: 0.1936
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction

## Train runs with `selection_metric = full_valid_ndcg@5`


### anchor_pd_s42


In [23]:
!python src/train_sasrec.py \
  --run_name anchor_pd_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 200 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_pd_s42
epoch=1, loss=0.4974
epoch=2, loss=0.2282
epoch=3, loss=0.1655
epoch=4, loss=0.1360
epoch=5, loss=0.1199
valid [full], NDCG@5: 0.6457, HR@5: 0.7414, NDCG@10: 0.7226, HR@10: 0.9828, MRR: 0.6469
valid [sampled], NDCG@5: 0.5632, HR@5: 0.5772, NDCG@10: 0.5810, HR@10: 0.6324, MRR: 0.5772
test [full], NDCG@5: 0.5359, HR@5: 0.7313, NDCG@10: 0.6238, HR@10: 1.0000, MRR: 0.5074
test [sampled], NDCG@5: 0.1698, HR@5: 0.1750, NDCG@10: 0.2127, HR@10: 0.3140, MRR: 0.2129
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sa

### anchor_ml100_s42


In [24]:
!python src/train_sasrec.py \
  --run_name anchor_ml100_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 100 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_ml100_s42
epoch=1, loss=0.4891
epoch=2, loss=0.2275
epoch=3, loss=0.1620
epoch=4, loss=0.1320
epoch=5, loss=0.1166
valid [full], NDCG@5: 0.6230, HR@5: 0.6725, NDCG@10: 0.7174, HR@10: 0.9646, MRR: 0.6482
valid [sampled], NDCG@5: 0.5689, HR@5: 0.5814, NDCG@10: 0.5844, HR@10: 0.6298, MRR: 0.5812
test [full], NDCG@5: 0.4660, HR@5: 0.6063, NDCG@10: 0.5864, HR@10: 0.9842, MRR: 0.4681
test [sampled], NDCG@5: 0.1427, HR@5: 0.1480, NDCG@10: 0.1838, HR@10: 0.2807, MRR: 0.1835
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs

### anchor_ml50_s42


In [25]:
!python src/train_sasrec.py \
  --run_name anchor_ml50_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_ml50_s42
epoch=1, loss=0.4793
epoch=2, loss=0.2166
epoch=3, loss=0.1564
epoch=4, loss=0.1245
epoch=5, loss=0.1127
valid [full], NDCG@5: 0.6566, HR@5: 0.7705, NDCG@10: 0.7221, HR@10: 0.9808, MRR: 0.6467
valid [sampled], NDCG@5: 0.5561, HR@5: 0.5710, NDCG@10: 0.5758, HR@10: 0.6326, MRR: 0.5713
test [full], NDCG@5: 0.4875, HR@5: 0.6521, NDCG@10: 0.5959, HR@10: 0.9999, MRR: 0.4741
test [sampled], NDCG@5: 0.1370, HR@5: 0.1437, NDCG@10: 0.1829, HR@10: 0.2915, MRR: 0.1796
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/

### anchor_ml20_s42


In [26]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/anchor_ml20_s42
epoch=1, loss=0.6703
epoch=2, loss=0.3156
epoch=3, loss=0.2178
epoch=4, loss=0.1735
epoch=5, loss=0.1464
valid [full], NDCG@5: 0.7050, HR@5: 0.8805, NDCG@10: 0.7424, HR@10: 0.9916, MRR: 0.6648
valid [sampled], NDCG@5: 0.5452, HR@5: 0.5520, NDCG@10: 0.5650, HR@10: 0.6151, MRR: 0.5682
test [full], NDCG@5: 0.8392, HR@5: 0.9371, NDCG@10: 0.8599, HR@10: 1.0000, MRR: 0.8146
test [sampled], NDCG@5: 0.1971, HR@5: 0.2797, NDCG@10: 0.2821, HR@10: 0.5416, MRR: 0.2314
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/

### refine_ml50_do030_s42


In [27]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do030_s42
epoch=1, loss=0.5360
epoch=2, loss=0.2475
epoch=3, loss=0.1849
epoch=4, loss=0.1493
epoch=5, loss=0.1305
valid [full], NDCG@5: 0.6477, HR@5: 0.7576, NDCG@10: 0.6943, HR@10: 0.9076, MRR: 0.6380
valid [sampled], NDCG@5: 0.5544, HR@5: 0.5666, NDCG@10: 0.5702, HR@10: 0.6159, MRR: 0.5690
test [full], NDCG@5: 0.4378, HR@5: 0.5577, NDCG@10: 0.5739, HR@10: 0.9914, MRR: 0.4509
test [sampled], NDCG@5: 0.1360, HR@5: 0.1425, NDCG@10: 0.1793, HR@10: 0.2819, MRR: 0.1759
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/ou

### refine_ml50_do035_s42


In [28]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do035_s42
epoch=1, loss=0.5673
epoch=2, loss=0.2638
epoch=3, loss=0.1987
epoch=4, loss=0.1638
epoch=5, loss=0.1414
valid [full], NDCG@5: 0.6434, HR@5: 0.7559, NDCG@10: 0.6854, HR@10: 0.8896, MRR: 0.6333
valid [sampled], NDCG@5: 0.5545, HR@5: 0.5603, NDCG@10: 0.5654, HR@10: 0.5947, MRR: 0.5698
test [full], NDCG@5: 0.4390, HR@5: 0.6201, NDCG@10: 0.5094, HR@10: 0.8432, MRR: 0.4211
test [sampled], NDCG@5: 0.1322, HR@5: 0.1359, NDCG@10: 0.1472, HR@10: 0.1844, MRR: 0.1664
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/ou

### refine_ml50_do025_s42


In [29]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do025_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do025_s42
epoch=1, loss=0.5066
epoch=2, loss=0.2315
epoch=3, loss=0.1700
epoch=4, loss=0.1361
epoch=5, loss=0.1203
valid [full], NDCG@5: 0.6551, HR@5: 0.7708, NDCG@10: 0.7204, HR@10: 0.9803, MRR: 0.6446
valid [sampled], NDCG@5: 0.5541, HR@5: 0.5661, NDCG@10: 0.5730, HR@10: 0.6257, MRR: 0.5705
test [full], NDCG@5: 0.4666, HR@5: 0.6195, NDCG@10: 0.5856, HR@10: 1.0000, MRR: 0.4610
test [sampled], NDCG@5: 0.1341, HR@5: 0.1405, NDCG@10: 0.1812, HR@10: 0.2923, MRR: 0.1766
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/ou

### refine_ml50_do025_s2024


In [30]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do025_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do025_s2024
epoch=1, loss=0.5681
epoch=2, loss=0.2402
epoch=3, loss=0.1763
epoch=4, loss=0.1409
epoch=5, loss=0.1248
valid [full], NDCG@5: 0.6843, HR@5: 0.8465, NDCG@10: 0.7338, HR@10: 0.9939, MRR: 0.6549
valid [sampled], NDCG@5: 0.5604, HR@5: 0.5637, NDCG@10: 0.5685, HR@10: 0.5894, MRR: 0.5784
test [full], NDCG@5: 0.7561, HR@5: 0.8822, NDCG@10: 0.7931, HR@10: 1.0000, MRR: 0.7288
test [sampled], NDCG@5: 0.1681, HR@5: 0.2351, NDCG@10: 0.2364, HR@10: 0.4463, MRR: 0.2037
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml50_do030_s2024


In [31]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do030_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml50_do030_s2024
epoch=1, loss=0.5987
epoch=2, loss=0.2553
epoch=3, loss=0.1894
epoch=4, loss=0.1533
epoch=5, loss=0.1350
valid [full], NDCG@5: 0.7439, HR@5: 0.9658, NDCG@10: 0.7536, HR@10: 0.9931, MRR: 0.6785
valid [sampled], NDCG@5: 0.5670, HR@5: 0.5725, NDCG@10: 0.5807, HR@10: 0.6159, MRR: 0.5882
test [full], NDCG@5: 0.7688, HR@5: 0.9350, NDCG@10: 0.7900, HR@10: 1.0000, MRR: 0.7235
test [sampled], NDCG@5: 0.1627, HR@5: 0.2252, NDCG@10: 0.2248, HR@10: 0.4173, MRR: 0.1993
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

### refine_ml75_do030_s42


In [32]:
!python src/train_sasrec.py \
  --run_name refine_ml75_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 75 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml75_do030_s42
epoch=1, loss=0.5423
epoch=2, loss=0.2566
epoch=3, loss=0.1895
epoch=4, loss=0.1550
epoch=5, loss=0.1346
valid [full], NDCG@5: 0.6334, HR@5: 0.7080, NDCG@10: 0.7187, HR@10: 0.9788, MRR: 0.6444
valid [sampled], NDCG@5: 0.5632, HR@5: 0.5782, NDCG@10: 0.5810, HR@10: 0.6336, MRR: 0.5759
test [full], NDCG@5: 0.4215, HR@5: 0.5434, NDCG@10: 0.5647, HR@10: 0.9999, MRR: 0.4357
test [sampled], NDCG@5: 0.1462, HR@5: 0.1517, NDCG@10: 0.1850, HR@10: 0.2766, MRR: 0.1841
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/ou

### refine_ml100_do030_s42


In [33]:
!python src/train_sasrec.py \
  --run_name refine_ml100_do030_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 100 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --seed 42 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5/refine_ml100_do030_s42
epoch=1, loss=0.5443
epoch=2, loss=0.2581
epoch=3, loss=0.1902
epoch=4, loss=0.1567
epoch=5, loss=0.1356
valid [full], NDCG@5: 0.6461, HR@5: 0.7348, NDCG@10: 0.7279, HR@10: 0.9925, MRR: 0.6507
valid [sampled], NDCG@5: 0.5705, HR@5: 0.5853, NDCG@10: 0.5859, HR@10: 0.6330, MRR: 0.5827
test [full], NDCG@5: 0.4594, HR@5: 0.5923, NDCG@10: 0.5847, HR@10: 0.9859, MRR: 0.4657
test [sampled], NDCG@5: 0.1540, HR@5: 0.1590, NDCG@10: 0.1933, HR@10: 0.2859, MRR: 0.1936
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

## Result lookup

Rebuild result tables directly from each run folder to avoid any CSV schema drift.


In [4]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [5]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)


## Result lookup: ndcg10


In [10]:
df_ndcg10 = rebuild_df(NDCG10_OUTPUT_DIR)
# df_ndcg10 = df_ndcg10.sort_values(['best_valid_full_ndcg@10', 'best_valid_full_hr@10'], ascending=False).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'maxlen', 'dropout_rate', 'best_epoch',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
    # 'checkpoint_best', 'checkpoint_last',
]]


,run_name,seed,maxlen,dropout_rate,best_epoch,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,anchor_pd_s42,42,200,0.20,30,0.732820,0.976377,0.687751,0.837473,0.660809,0.759349,1.0,0.741494,0.945437,0.677949,0.571892,0.608637,0.559131,0.568577,0.576002,0.266000,0.425584,0.198901,0.209418,0.257410
1,anchor_ml100_s42,42,100,0.20,30,0.731288,0.976377,0.664780,0.765794,0.660439,0.805918,1.0,0.780413,0.919929,0.741519,0.581806,0.636746,0.562553,0.576725,0.577579,0.312154,0.508973,0.238265,0.275536,0.284821
2,anchor_ml50_s42,42,50,0.20,15,0.732238,0.977745,0.694772,0.860632,0.658512,0.821952,1.0,0.797110,0.920622,0.764088,0.575181,0.621538,0.559087,0.571274,0.576963,0.291792,0.516651,0.218746,0.290064,0.252954
3,anchor_ml20_s42,42,20,0.20,50,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.0,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
4,refine_ml50_do030_s42,42,50,0.30,15,0.728433,0.975302,0.683934,0.833085,0.655886,0.843098,1.0,0.819527,0.927924,0.792374,0.585535,0.628564,0.570842,0.582894,0.586061,0.294100,0.541253,0.215080,0.297268,0.246889
5,refine_ml50_do035_s42,42,50,0.35,50,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.0,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
6,refine_ml50_do025_s42,42,50,0.25,15,0.737294,0.975302,0.698773,0.855883,0.666429,0.757232,1.0,0.733023,0.922650,0.676185,0.590335,0.630861,0.576735,0.588839,0.592687,0.243580,0.427348,0.171452,0.197227,0.225910
7,refine_ml50_do025_s2024,2024,50,0.25,5,0.733755,0.993919,0.684304,0.846486,0.654911,0.793137,1.0,0.756145,0.882210,0.728828,0.568471,0.589448,0.560388,0.563678,0.578420,0.236381,0.446283,0.168065,0.235079,0.203745
8,refine_ml50_do030_s2024,2024,50,0.30,5,0.753595,0.993108,0.743881,0.965811,0.678463,0.790040,1.0,0.768786,0.935026,0.723476,0.580711,0.615896,0.567026,0.572494,0.588228,0.224842,0.417254,0.162712,0.225176,0.199310
9,refine_ml75_do030_s42,42,75,0.30,40,0.727543,0.975913,0.689849,0.854533,0.653961,0.895066,1.0,0.870918,0.922094,0.862916,0.565326,0.594543,0.553512,0.556456,0.573899,0.380006,0.611456,0.308142,0.390165,0.332916


## Result lookup: ndcg5


In [11]:
df_ndcg5 = rebuild_df(NDCG5_OUTPUT_DIR)
# df_ndcg5 = df_ndcg5.sort_values(['best_valid_full_ndcg@5', 'best_valid_full_hr@5'], ascending=False).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'maxlen', 'dropout_rate', 'best_epoch',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
    # 'checkpoint_best', 'checkpoint_last',
]]


,run_name,seed,maxlen,dropout_rate,best_epoch,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,anchor_pd_s42,42,200,0.20,10,0.729260,0.975289,0.697986,0.875628,0.655816,0.715432,1.0,0.680432,0.895188,0.622826,0.572131,0.611314,0.559202,0.571525,0.575570,0.228458,0.374611,0.167941,0.180945,0.221445
1,anchor_ml100_s42,42,100,0.20,45,0.725856,0.955607,0.681395,0.816725,0.659588,0.777539,1.0,0.750338,0.913079,0.706159,0.573727,0.622219,0.555278,0.564433,0.573421,0.337994,0.425760,0.305262,0.321254,0.345565
2,anchor_ml50_s42,42,50,0.20,15,0.732238,0.977745,0.694772,0.860632,0.658512,0.821952,1.0,0.797110,0.920622,0.764088,0.575181,0.621538,0.559087,0.571274,0.576963,0.291792,0.516651,0.218746,0.290064,0.252954
3,anchor_ml20_s42,42,20,0.20,50,0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.0,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
4,refine_ml50_do030_s42,42,50,0.30,15,0.728433,0.975302,0.683934,0.833085,0.655886,0.843098,1.0,0.819527,0.927924,0.792374,0.585535,0.628564,0.570842,0.582894,0.586061,0.294100,0.541253,0.215080,0.297268,0.246889
5,refine_ml50_do035_s42,42,50,0.35,50,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.0,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
6,refine_ml50_do025_s42,42,50,0.25,15,0.737294,0.975302,0.698773,0.855883,0.666429,0.757232,1.0,0.733023,0.922650,0.676185,0.590335,0.630861,0.576735,0.588839,0.592687,0.243580,0.427348,0.171452,0.197227,0.225910
7,refine_ml50_do025_s2024,2024,50,0.25,15,0.732527,0.974495,0.694564,0.855922,0.659681,0.881074,1.0,0.854356,0.916249,0.844470,0.563718,0.607806,0.546371,0.551972,0.568552,0.314189,0.587908,0.225346,0.313000,0.255963
8,refine_ml50_do030_s2024,2024,50,0.30,5,0.753595,0.993108,0.743881,0.965811,0.678463,0.790040,1.0,0.768786,0.935026,0.723476,0.580711,0.615896,0.567026,0.572494,0.588228,0.224842,0.417254,0.162712,0.225176,0.199310
9,refine_ml75_do030_s42,42,75,0.30,40,0.727543,0.975913,0.689849,0.854533,0.653961,0.895066,1.0,0.870918,0.922094,0.862916,0.565326,0.594543,0.553512,0.556456,0.573899,0.380006,0.611456,0.308142,0.390165,0.332916
